# Python Deep Agents 复杂业务案例 2：电商售后异常工单处理

这个 notebook 是第二个 Deep Agents 示例，业务场景和采购版不同。

它模拟一个电商售后团队处理复杂工单：客户反馈商品破损、未收到货、想退货，系统需要综合订单、物流、库存、客户风险、售后政策和 SLA，给出处理方案。

学习路径仍然是递进式：

```text
第 1 层：不用 Agent，普通 Python 跑通售后决策链路
第 2 层：把完整售后评审封装成 1 个工具，交给 Deep Agent 总结
第 3 层：拆成多个工具，让 Deep Agent 自己编排
第 4 层：加入子 Agent，让“物流/风控/客服主管”分工复核
第 5 层：让 Agent 写出处理报告和客户回复草稿
```

这个案例比采购版复杂一些，但每一步都能单独执行和观察。


## 1. 安装与环境检查

如果依赖已经安装，这一节只会打印“已安装”。


In [1]:
import importlib.util
import subprocess
import sys


def ensure_package(import_name: str, pip_name: str | None = None) -> None:
    """如果当前 Python 环境缺少依赖，就自动安装。"""
    if importlib.util.find_spec(import_name):
        print(f"{import_name} 已安装")
        return

    package = pip_name or import_name
    print(f"正在安装 {package} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])


ensure_package("deepagents")
ensure_package("langchain_openai", "langchain-openai")
ensure_package("dotenv", "python-dotenv")

print("依赖检查完成。")


deepagents 已安装
langchain_openai 已安装
dotenv 已安装
依赖检查完成。


## 2. 读取 `.env` 并创建 DeepSeek 模型

注意：这里默认使用 DeepSeek 官方地址 `https://api.deepseek.com`。

不会默认读取 `OPENAI_BASE_URL`，避免拿到本机代理或其他兼容接口地址。


In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI


def find_project_env(start: Path | None = None) -> Path:
    """从当前目录向上查找项目根目录 .env。"""
    current = (start or Path.cwd()).resolve()
    for directory in [current, *current.parents]:
        env_path = directory / ".env"
        if env_path.exists():
            return env_path
    raise FileNotFoundError(f"从 {current} 向上没有找到 .env。")


ENV_PATH = find_project_env()
PROJECT_ROOT = ENV_PATH.parent
load_dotenv(ENV_PATH, override=True)
print(f"已加载 .env: {ENV_PATH}")


def build_deepseek_chat_model() -> ChatOpenAI:
    """创建 DeepSeek Chat 模型。"""
    api_key = os.getenv("DEEPSEEK_API_KEY")
    if not api_key:
        raise RuntimeError("请先在项目 .env 中配置 DEEPSEEK_API_KEY。")

    base_url = os.getenv("DEEPSEEK_BASE_URL") or "https://api.deepseek.com"
    if base_url.rstrip("/").endswith("/anthropic"):
        base_url = base_url.rstrip("/")[: -len("/anthropic")]

    model = os.getenv("DEEPSEEK_MODEL") or os.getenv("OPENAI_MODEL") or "deepseek-chat"
    print(f"DeepSeek Chat: model={model}, base_url={base_url}")

    return ChatOpenAI(
        model=model,
        api_key=api_key,
        base_url=base_url,
        temperature=0,
        timeout=float(os.getenv("DEEPSEEK_TIMEOUT", "120")),
        max_retries=int(os.getenv("DEEPSEEK_MAX_RETRIES", "2")),
    )


llm = build_deepseek_chat_model()
response = llm.invoke("只回答两个字：成功")
print("模型返回：", response.content)


已加载 .env: D:\PythonProject\LearnOne\.env
DeepSeek Chat: model=deepseek-v4-flash, base_url=https://api.deepseek.com
模型返回： 成功


## 3. 加载售后业务数据

数据文件路径：

```text
docs/sample_docs/after_sales_business_data.json
```

它模拟这些业务系统：

| 数据模块 | 模拟系统 |
| --- | --- |
| `tickets` | 售后工单系统 |
| `orders` | 订单系统 |
| `logistics` | 物流轨迹系统 |
| `customers` | 会员和风控系统 |
| `inventory` | 库存系统 |
| `policies` | 售后政策库 |
| `sla_rules` | SLA 配置 |


In [3]:
import json
from datetime import datetime
from pprint import pprint


DATA_PATH = PROJECT_ROOT / "docs" / "sample_docs" / "after_sales_business_data.json"
after_sales_data = json.loads(DATA_PATH.read_text(encoding="utf-8"))

print("业务数据文件:", DATA_PATH)
print("数据模块:", list(after_sales_data.keys()))
print("\n示例工单:")
pprint(after_sales_data["tickets"][0], width=120)


业务数据文件: D:\PythonProject\LearnOne\docs\sample_docs\after_sales_business_data.json
数据模块: ['tickets', 'orders', 'logistics', 'customers', 'inventory', 'policies', 'sla_rules']

示例工单:
{'channel': 'app',
 'created_at': '2026-07-10 09:12:00',
 'customer_id': 'C-10086',
 'customer_message': '我买的智能咖啡机到货后发现外壳裂了，水箱也漏水。已经影响使用，希望今天给我一个处理结果。',
 'order_id': 'OD-2026-0628-9001',
 'priority': 'high',
 'ticket_id': 'AS-2026-0710-001'}


## 4. 定义基础查询工具

先别看 Agent。

下面这些函数就是企业内部 API 的模拟版：

```text
get_ticket        查工单
get_order         查订单
get_logistics     查物流
get_customer      查客户风险
get_inventory     查库存
search_policy     查售后政策
get_sla_rule      查 SLA
```

它们都返回 JSON 字符串，因为 Agent 读取 JSON 比读取随意文本稳定。


In [4]:
def pretty_json(data) -> str:
    """把 dict/list/JSON 字符串格式化，方便 notebook 查看。"""
    if isinstance(data, str):
        data = json.loads(data)
    return json.dumps(data, ensure_ascii=False, indent=2)


def find_one(rows: list[dict], key: str, value: str) -> dict | None:
    """在列表里按字段查一条记录。"""
    for row in rows:
        if row.get(key) == value:
            return row
    return None


def get_ticket(ticket_id: str) -> str:
    """查询售后工单。"""
    row = find_one(after_sales_data["tickets"], "ticket_id", ticket_id)
    return json.dumps(row or {"ticket_id": ticket_id, "found": False}, ensure_ascii=False)


def get_order(order_id: str) -> str:
    """查询订单信息。"""
    row = find_one(after_sales_data["orders"], "order_id", order_id)
    return json.dumps(row or {"order_id": order_id, "found": False}, ensure_ascii=False)


def get_logistics(order_id: str) -> str:
    """查询物流签收、破损和凭证信息。"""
    row = find_one(after_sales_data["logistics"], "order_id", order_id)
    return json.dumps(row or {"order_id": order_id, "found": False}, ensure_ascii=False)


def get_customer(customer_id: str) -> str:
    """查询客户等级、退款次数、投诉次数和风险等级。"""
    row = find_one(after_sales_data["customers"], "customer_id", customer_id)
    return json.dumps(row or {"customer_id": customer_id, "found": False}, ensure_ascii=False)


def get_inventory(sku: str) -> str:
    """查询 SKU 可换新库存。"""
    row = find_one(after_sales_data["inventory"], "sku", sku)
    return json.dumps(row or {"sku": sku, "found": False}, ensure_ascii=False)


def search_policy(keyword: str) -> str:
    """按关键词检索售后政策。演示版使用简单包含匹配。"""
    hits = []
    for policy in after_sales_data["policies"]:
        text = policy["name"] + " " + policy["rule"]
        if keyword in text:
            hits.append(policy)
    return json.dumps({"keyword": keyword, "hits": hits}, ensure_ascii=False)


def get_sla_rule(priority: str) -> str:
    """按工单优先级查询 SLA。"""
    row = find_one(after_sales_data["sla_rules"], "priority", priority)
    return json.dumps(row or {"priority": priority, "found": False}, ensure_ascii=False)


print("基础查询工具定义完成。")


基础查询工具定义完成。


## 5. 第 1 层：手动处理一个复杂售后工单

先处理第一个工单：客户收到智能咖啡机后发现破损漏水。

手动链路如下：

```text
工单 -> 订单 -> 物流 -> 客户风险 -> 库存 -> 政策 -> SLA -> 决策
```

这一节不使用 Agent，目的只是让你看清楚业务数据怎么流动。


In [5]:
ticket_id = "AS-2026-0710-001"

ticket = json.loads(get_ticket(ticket_id))
order = json.loads(get_order(ticket["order_id"]))
logistics = json.loads(get_logistics(order["order_id"]))
customer = json.loads(get_customer(ticket["customer_id"]))
inventory = json.loads(get_inventory(order["sku"]))
policy_damage = json.loads(search_policy("破损"))
policy_high_value = json.loads(search_policy("高价值"))
sla = json.loads(get_sla_rule(ticket["priority"]))

print("1. 工单")
print(pretty_json(ticket))
print("\n2. 订单")
print(pretty_json(order))
print("\n3. 物流")
print(pretty_json(logistics))
print("\n4. 客户")
print(pretty_json(customer))
print("\n5. 库存")
print(pretty_json(inventory))
print("\n6. 相关政策")
print(pretty_json({"damage_policy": policy_damage, "high_value_policy": policy_high_value}))
print("\n7. SLA")
print(pretty_json(sla))


1. 工单
{
  "ticket_id": "AS-2026-0710-001",
  "customer_id": "C-10086",
  "order_id": "OD-2026-0628-9001",
  "channel": "app",
  "priority": "high",
  "created_at": "2026-07-10 09:12:00",
  "customer_message": "我买的智能咖啡机到货后发现外壳裂了，水箱也漏水。已经影响使用，希望今天给我一个处理结果。"
}

2. 订单
{
  "order_id": "OD-2026-0628-9001",
  "customer_id": "C-10086",
  "sku": "SKU-COFFEE-PRO-01",
  "item_name": "Barista Pro 智能咖啡机",
  "quantity": 1,
  "paid_amount": 3299,
  "paid_at": "2026-06-28 20:18:00",
  "payment_method": "credit_card",
  "warranty_months": 12,
  "return_window_days": 7,
  "delivery_promise": "2026-07-03",
  "seller": "自营旗舰店"
}

3. 物流
{
  "order_id": "OD-2026-0628-9001",
  "tracking_no": "SF99881230001",
  "carrier": "顺丰",
  "status": "delivered",
  "delivered_at": "2026-07-02 16:40:00",
  "signed_by": "本人签收",
  "damage_report": true,
  "damage_note": "派送员备注：外箱一角明显挤压，客户当场拍照反馈。",
  "proof_images": [
    "outer_box_crushed.jpg",
    "water_tank_leak.jpg"
  ]
}

4. 客户
{
  "customer_id": "C-10086",
  "name":

## 6. 定义售后决策函数

这一节把业务规则写成确定性函数。

它会判断：

- 问题类型：破损、未收到货、七天无理由等。
- 是否可以换新。
- 是否可以退款。
- 是否需要物流核查。
- 是否需要主管审批。
- 建议给客户什么回复。

这里仍然不是 Agent，是普通 Python 业务代码。


In [6]:
def classify_issue(ticket: dict, logistics: dict) -> str:
    """根据客户描述和物流信息判断问题类型。"""
    message = ticket.get("customer_message", "")
    if logistics.get("damage_report") or "裂" in message or "漏水" in message or "破损" in message:
        return "到货破损"
    if "没收到" in message or "没有收到" in message:
        return "未收到货"
    if "退货" in message or "退款" in message or "不喜欢" in message:
        return "七天无理由退货"
    return "一般售后咨询"


def days_between(left: str, right: str) -> int:
    """计算两个 yyyy-mm-dd hh:mm:ss 时间相差天数。"""
    left_dt = datetime.strptime(left, "%Y-%m-%d %H:%M:%S")
    right_dt = datetime.strptime(right, "%Y-%m-%d %H:%M:%S")
    return abs((right_dt - left_dt).days)


def make_after_sales_decision(
    ticket: dict,
    order: dict,
    logistics: dict,
    customer: dict,
    inventory: dict,
    sla: dict,
) -> dict:
    """根据业务事实生成售后处理决策。"""
    issue_type = classify_issue(ticket, logistics)
    paid_amount = order.get("paid_amount", 0)
    high_value = paid_amount > 2000
    customer_risk = customer.get("risk_level", "unknown")
    new_stock = inventory.get("available_new", 0)
    delivered_at = logistics.get("delivered_at")
    created_at = ticket.get("created_at")
    days_after_delivery = days_between(delivered_at, created_at) if delivered_at and created_at else None

    # 售后系统正式建单可能晚于签收现场反馈。
    # 如果物流备注里已经记录“客户当场拍照反馈”，就视为存在及时破损证据。
    timely_damage_evidence = bool(
        logistics.get("damage_report")
        and (
            (days_after_delivery is not None and days_after_delivery <= 3)
            or "当场" in logistics.get("damage_note", "")
        )
    )

    actions = []
    blockers = []
    approvals = []
    customer_reply_points = []
    compensation = {"type": "none", "amount": 0, "reason": ""}

    if issue_type == "到货破损":
        if timely_damage_evidence:
            if new_stock > 0:
                actions.append("创建换新单，优先从可用新库存发出。")
                customer_reply_points.append("可为客户安排换新。")
            else:
                actions.append("库存不足，提供原路退款或等待补货换新。")
                customer_reply_points.append("当前无可换新库存，可选择退款或等待补货。")
            compensation = {"type": "coupon", "amount": 80, "reason": "物流破损影响体验"}
        else:
            blockers.append("破损反馈不满足 72 小时或缺少物流破损证据，需要人工复核。")
    elif issue_type == "未收到货":
        actions.append("发起物流核查，要求承运商提供签收凭证。")
        customer_reply_points.append("已发起物流核查，核查完成前先不直接退款。")
        if customer_risk in {"medium", "high"}:
            blockers.append("客户风险等级非低，物流显示签收，不能直接退款。")
    elif issue_type == "七天无理由退货":
        if days_after_delivery is not None and days_after_delivery <= order.get("return_window_days", 7):
            actions.append("创建退货退款单，客户寄回后验货退款。")
            customer_reply_points.append("符合七天无理由退货窗口。")
        else:
            blockers.append("已超过七天无理由退货窗口。")
    else:
        actions.append("转人工客服进一步确认诉求。")

    if high_value and ("退款" in " ".join(actions) or "换新" in " ".join(actions) or compensation["amount"] > 0):
        approvals.append("售后主管审批：高价值订单涉及退款/换新/补偿。")
    if compensation["amount"] > 100:
        approvals.append("售后主管审批：现金或高额补偿超过 100 元。")

    if blockers:
        recommendation = "暂缓自动处理，先补充核查或人工审批。"
    elif approvals:
        recommendation = "建议通过，但需要主管审批后执行。"
    else:
        recommendation = "建议自动处理。"

    return {
        "issue_type": issue_type,
        "days_after_delivery": days_after_delivery,
        "timely_damage_evidence": timely_damage_evidence,
        "high_value_order": high_value,
        "customer_risk": customer_risk,
        "actions": actions,
        "blockers": blockers,
        "approvals": approvals,
        "compensation": compensation,
        "sla": sla,
        "customer_reply_points": customer_reply_points,
        "recommendation": recommendation,
    }


decision = make_after_sales_decision(ticket, order, logistics, customer, inventory, sla)
print(pretty_json(decision))


{
  "issue_type": "到货破损",
  "days_after_delivery": 7,
  "timely_damage_evidence": true,
  "high_value_order": true,
  "customer_risk": "low",
  "actions": [
    "创建换新单，优先从可用新库存发出。"
  ],
  "blockers": [],
  "approvals": [
    "售后主管审批：高价值订单涉及退款/换新/补偿。"
  ],
  "compensation": {
    "type": "coupon",
    "amount": 80,
    "reason": "物流破损影响体验"
  },
  "sla": {
    "priority": "high",
    "first_response_minutes": 30,
    "resolution_hours": 8
  },
  "customer_reply_points": [
    "可为客户安排换新。"
  ],
  "recommendation": "建议通过，但需要主管审批后执行。"
}


## 7. 把完整链路封装成一个大工具

真实项目里，很多关键业务流程不应该完全交给 LLM 自由发挥。

更稳的做法是：

```text
确定性代码负责查数和判断
LLM 负责解释、总结、生成回复草稿
```

所以下面封装一个大工具：`run_after_sales_review`。


In [7]:
def run_after_sales_review(ticket_id: str) -> str:
    """完整售后工单评审工具。

    输入：售后工单 ID。
    输出：JSON 字符串，包含所有业务事实、决策、政策和建议。
    """
    ticket = json.loads(get_ticket(ticket_id))
    if not ticket or ticket.get("found") is False:
        return json.dumps({"ticket_id": ticket_id, "error": "工单不存在"}, ensure_ascii=False)

    order = json.loads(get_order(ticket["order_id"]))
    logistics = json.loads(get_logistics(order["order_id"]))
    customer = json.loads(get_customer(ticket["customer_id"]))
    inventory = json.loads(get_inventory(order["sku"]))
    sla = json.loads(get_sla_rule(ticket["priority"]))

    issue_type = classify_issue(ticket, logistics)
    policy_keywords = {
        "到货破损": ["破损", "高价值", "补偿"],
        "未收到货": ["未收到货"],
        "七天无理由退货": ["七天无理由"],
    }.get(issue_type, [])
    policies = [json.loads(search_policy(keyword)) for keyword in policy_keywords]

    decision = make_after_sales_decision(ticket, order, logistics, customer, inventory, sla)

    result = {
        "ticket": ticket,
        "order": order,
        "logistics": logistics,
        "customer": customer,
        "inventory": inventory,
        "policies": policies,
        "decision": decision,
    }
    return json.dumps(result, ensure_ascii=False)


review_json = run_after_sales_review(ticket_id)
print(pretty_json(review_json))


{
  "ticket": {
    "ticket_id": "AS-2026-0710-001",
    "customer_id": "C-10086",
    "order_id": "OD-2026-0628-9001",
    "channel": "app",
    "priority": "high",
    "created_at": "2026-07-10 09:12:00",
    "customer_message": "我买的智能咖啡机到货后发现外壳裂了，水箱也漏水。已经影响使用，希望今天给我一个处理结果。"
  },
  "order": {
    "order_id": "OD-2026-0628-9001",
    "customer_id": "C-10086",
    "sku": "SKU-COFFEE-PRO-01",
    "item_name": "Barista Pro 智能咖啡机",
    "quantity": 1,
    "paid_amount": 3299,
    "paid_at": "2026-06-28 20:18:00",
    "payment_method": "credit_card",
    "warranty_months": 12,
    "return_window_days": 7,
    "delivery_promise": "2026-07-03",
    "seller": "自营旗舰店"
  },
  "logistics": {
    "order_id": "OD-2026-0628-9001",
    "tracking_no": "SF99881230001",
    "carrier": "顺丰",
    "status": "delivered",
    "delivered_at": "2026-07-02 16:40:00",
    "signed_by": "本人签收",
    "damage_report": true,
    "damage_note": "派送员备注：外箱一角明显挤压，客户当场拍照反馈。",
    "proof_images": [
      "outer_box_crushed.

## 8. 第 2 层：最小 Deep Agent，只调用一个大工具

现在引入 Deep Agent。

它只拿到一个工具：`run_after_sales_review`。

Agent 负责：

```text
调用工具拿到事实 -> 解释处理建议 -> 生成客服可读结论
```


In [8]:
from deepagents import create_deep_agent


simple_after_sales_agent = create_deep_agent(
    model=llm,
    tools=[run_after_sales_review],
    system_prompt=(
        "你是电商售后工单处理助手。"
        "收到工单 ID 后，必须调用 run_after_sales_review 获取真实业务结果。"
        "最后用中文输出：问题类型、处理建议、是否需要审批、客户回复要点、SLA。"
    ),
)

simple_result = simple_after_sales_agent.invoke({"messages": [{"role": "user", "content": f"请处理工单 {ticket_id}"}]})
print(simple_result["messages"][-1].content)


## 工单 AS-2026-0710-001 处理结果

### 📋 问题类型
**到货破损** — 客户李女士购买的 Barista Pro 智能咖啡机（SKU-COFFEE-PRO-01，金额 3299 元）到货后发现外壳裂开、水箱漏水。物流外箱有明显挤压痕迹，客户当场拍照，且在签收后 72 小时内反馈，属于完整的物流破损举证。

---

### 💡 处理建议
1. **优先换新** — 华东仓目前有 3 台新库存可用，建议立即创建换新单，从新库存发出。
2. **客户体验补偿** — 因物流破损影响客户体验，建议补偿 80 元优惠券。
3. **退款兜底** — 如换新过程中出现异常，可按照政策原路退款并补偿不超过订单金额的 5%（约 165 元）。

---

### ✅ 是否需要审批
**是，需要售后主管审批。** 原因：
- 订单金额 3299 元 > 2000 元，属于高价值订单，涉及换新/补偿需主管审批（P004）。
- 现金补偿若超过 100 元需主管审批，本次补偿为 80 元优惠券，在免审批额度内。

---

### 📝 客户回复要点
1. 可为客户安排 **换新**，优先从华东仓新库存发出。
2. 因物流破损影响使用体验，额外补偿 **80 元优惠券**。
3. 致歉并告知换新预计时效。

---

### ⏱ SLA

| 指标 | 值 |
|------|-----|
| 优先级 | 🔴 **高** |
| 首次响应时限 | 30 分钟内 |
| 解决时效 | 8 小时内 |

> 客户已明确要求"今天给处理结果"，请在 SLA 时限内尽快完成审批并答复。


## 9. 查看 Agent 工具调用轨迹

这里验证 Agent 是否真的调用了工具。


In [9]:
def print_agent_trace(agent_result: dict, max_chars: int = 1800) -> None:
    """打印 AI 消息和工具消息，观察 Agent 是如何执行的。"""
    for index, message in enumerate(agent_result.get("messages", []), start=1):
        msg_type = message.__class__.__name__
        content = getattr(message, "content", "")
        tool_calls = getattr(message, "tool_calls", None)
        if tool_calls or msg_type in {"ToolMessage", "AIMessage"}:
            print("=" * 80)
            print(f"#{index} {msg_type}")
            if tool_calls:
                print("tool_calls:", tool_calls)
            print(str(content)[:max_chars])


print_agent_trace(simple_result)


#2 AIMessage
tool_calls: [{'name': 'run_after_sales_review', 'args': {'ticket_id': 'AS-2026-0710-001'}, 'id': 'call_00_KBMjA72DU8fQxyjCjBXc8524', 'type': 'tool_call'}]

#3 ToolMessage
{"ticket": {"ticket_id": "AS-2026-0710-001", "customer_id": "C-10086", "order_id": "OD-2026-0628-9001", "channel": "app", "priority": "high", "created_at": "2026-07-10 09:12:00", "customer_message": "我买的智能咖啡机到货后发现外壳裂了，水箱也漏水。已经影响使用，希望今天给我一个处理结果。"}, "order": {"order_id": "OD-2026-0628-9001", "customer_id": "C-10086", "sku": "SKU-COFFEE-PRO-01", "item_name": "Barista Pro 智能咖啡机", "quantity": 1, "paid_amount": 3299, "paid_at": "2026-06-28 20:18:00", "payment_method": "credit_card", "warranty_months": 12, "return_window_days": 7, "delivery_promise": "2026-07-03", "seller": "自营旗舰店"}, "logistics": {"order_id": "OD-2026-0628-9001", "tracking_no": "SF99881230001", "carrier": "顺丰", "status": "delivered", "delivered_at": "2026-07-02 16:40:00", "signed_by": "本人签收", "damage_report": true, "damage_note": "派送员备注：外箱一角明显挤压

## 10. 第 3 层：拆成多个工具，让 Agent 自己编排

这一层更复杂。

我们不再给 Agent 一个完整工具，而是给它所有小工具：

```text
get_ticket / get_order / get_logistics / get_customer / get_inventory / search_policy / get_sla_rule
```

它需要自己决定先查什么、后查什么。


In [10]:
multi_tool_after_sales_agent = create_deep_agent(
    model=llm,
    tools=[
        get_ticket,
        get_order,
        get_logistics,
        get_customer,
        get_inventory,
        search_policy,
        get_sla_rule,
    ],
    system_prompt=(
        "你是电商售后异常工单处理助手。必须按顺序完成："
        "1. get_ticket 查工单；"
        "2. get_order 查订单；"
        "3. get_logistics 查物流；"
        "4. get_customer 查客户风险；"
        "5. get_inventory 查库存；"
        "6. search_policy 查相关售后政策；"
        "7. get_sla_rule 查 SLA。"
        "最后输出结构化中文结论：问题类型、证据、处理动作、审批要求、客户回复草稿。"
    ),
)

multi_tool_result = multi_tool_after_sales_agent.invoke({"messages": [{"role": "user", "content": f"请处理工单 {ticket_id}"}]})
print(multi_tool_result["messages"][-1].content)


所有数据已收集完毕，以下是结构化处理结论。

---

## 工单处理结论

### 问题类型
**到货破损 — 物流挤压导致外壳开裂、水箱漏水**

### 证据
| 来源 | 内容 |
|---|---|
| 物流记录 | 2026-07-02 16:40 本人签收；派送员备注"外箱一角明显挤压，客户当场拍照反馈" |
| 凭证图片 | `outer_box_crushed.jpg`（外箱挤压）、`water_tank_leak.jpg`（水箱漏水） |
| 客户留言 | "外壳裂了，水箱漏水，影响使用" |
| 签收时间 | 2026-07-02，客户当场反馈 → 满足 **72小时内** 破损报备条件 |

### 客户画像
- 李女士，**铂金会员**，180天内18单，仅1次退款、0投诉，**低风险**
- 24小时内有8小时解决时效的紧迫诉求（SLA：首次响应 ≤30分钟，解决 ≤8小时）

### 处理动作
**建议：优先换新（全新机）**

1. **适用政策**：P001《到货破损处理》— 物流破损 + 72小时内反馈 → 优先换新
2. **库存充足**：华东仓现有 **3台全新机** 可发
3. **换新流程**：旧机通过顺丰上门取件退回，同时发出新机

### 审批要求
- **需售后主管审批**（政策 P004《高价值订单升级》）：订单金额 3,299元 > 2,000元，涉及换新需主管审批通过后执行

### 客户回复草稿
> 尊敬的李女士，您好：
>
> 非常抱歉给您带来不佳的体验。根据您提供的反馈照片及物流记录，确认商品在运输过程中因包装挤压导致外壳开裂和水箱漏水，符合我们的破损换新政策。
>
> 我们已为您安排**全新机换新**，预计今天内会生成换新单，届时会有顺丰快递员上门取回旧机（免费），同时发出新机。
>
> 感谢您的耐心与支持，如有任何疑问请随时联系我们。


## 11. 查看多工具 Agent 的调用轨迹

如果多工具 Agent 的结论不对，第一时间看 trace。

常见问题是：

- 漏查了库存。
- 没查 SLA。
- 政策关键词查错。
- 把工具返回的事实理解错。


In [11]:
print_agent_trace(multi_tool_result)


#2 AIMessage
tool_calls: [{'name': 'get_ticket', 'args': {'ticket_id': 'AS-2026-0710-001'}, 'id': 'call_00_8RTHPYLvynLpauYnsZNL4727', 'type': 'tool_call'}]

#3 ToolMessage
{"ticket_id": "AS-2026-0710-001", "customer_id": "C-10086", "order_id": "OD-2026-0628-9001", "channel": "app", "priority": "high", "created_at": "2026-07-10 09:12:00", "customer_message": "我买的智能咖啡机到货后发现外壳裂了，水箱也漏水。已经影响使用，希望今天给我一个处理结果。"}
#4 AIMessage
tool_calls: [{'name': 'get_order', 'args': {'order_id': 'OD-2026-0628-9001'}, 'id': 'call_00_Qv3J2jZv1sa7Wm2W7FF71115', 'type': 'tool_call'}]
工单信息：客户 C-10086 反馈智能咖啡机到货后外壳裂了、水箱漏水。接下来查订单信息。
#5 ToolMessage
{"order_id": "OD-2026-0628-9001", "customer_id": "C-10086", "sku": "SKU-COFFEE-PRO-01", "item_name": "Barista Pro 智能咖啡机", "quantity": 1, "paid_amount": 3299, "paid_at": "2026-06-28 20:18:00", "payment_method": "credit_card", "warranty_months": 12, "return_window_days": 7, "delivery_promise": "2026-07-03", "seller": "自营旗舰店"}
#6 AIMessage
tool_calls: [{'name': 'get_logistics'

## 12. 第 4 层：子 Agent 分工复核

复杂售后任务里，通常不是一个角色说了算：

- `logistics-investigator`：物流证据和破损责任。
- `risk-reviewer`：客户风险、是否能直接退款。
- `service-supervisor`：审批、补偿、客户沟通口径。

子 Agent 的价值是让不同角色有不同关注点。


In [12]:
subagents = [
    {
        "name": "logistics-investigator",
        "description": "负责核查物流签收、破损备注和凭证。",
        "system_prompt": "你是物流核查员。只关注物流状态、签收人、破损备注、凭证和责任判断。",
        "tools": [get_logistics],
    },
    {
        "name": "risk-reviewer",
        "description": "负责核查客户风险和退款风险。",
        "system_prompt": "你是售后风控复核员。只关注客户风险等级、历史退款投诉、是否可以直接退款。",
        "tools": [get_customer],
    },
    {
        "name": "service-supervisor",
        "description": "负责审批要求、补偿策略和客服沟通口径。",
        "system_prompt": "你是售后主管。只关注是否需要审批、补偿是否合理、客户回复是否清楚。",
        "tools": [search_policy, get_sla_rule],
    },
]

subagent_after_sales_agent = create_deep_agent(
    model=llm,
    tools=[run_after_sales_review],
    subagents=subagents,
    system_prompt=(
        "你是售后工单主控 Agent。"
        "优先调用 run_after_sales_review 获取完整事实和基础决策。"
        "必要时请子 Agent 复核物流、风控和主管审批意见。"
        "最终输出 Markdown 报告，包含事实、政策、处理动作、审批、客户回复草稿。"
    ),
)

subagent_result = subagent_after_sales_agent.invoke({"messages": [{"role": "user", "content": f"请处理工单 {ticket_id}"}]})
print(subagent_result["messages"][-1].content)


工单处理完毕，所有步骤均已闭环。


## 13. 第 5 层：写入报告和客户回复草稿

这一节让 Agent 使用虚拟文件系统写两个文件：

- `after_sales_resolution_report.md`：内部处理报告。
- `customer_reply_draft.md`：发给客户的回复草稿。


In [13]:
report_agent = create_deep_agent(
    model=llm,
    tools=[run_after_sales_review],
    system_prompt=(
        "你是售后工单报告助手。"
        "必须调用 run_after_sales_review 获取真实业务结果。"
        "然后写入两个 Markdown 文件："
        "1. after_sales_resolution_report.md：内部处理报告；"
        "2. customer_reply_draft.md：客户回复草稿。"
        "内部报告必须包含证据表、政策依据、处理动作、审批要求、SLA。"
        "客户回复要礼貌、明确、不要暴露内部风控字段。"
    ),
)

report_result = report_agent.invoke({"messages": [{"role": "user", "content": f"请处理工单 {ticket_id}，并生成报告和客户回复"}]})
print(report_result["messages"][-1].content)
print("\n虚拟文件系统 files:")
print(json.dumps(report_result.get("files", {}), ensure_ascii=False, indent=2)[:4000])


两份文件已生成完毕。以下是处理摘要：

---

### 工单 AS-2026-0710-001 处理摘要

| 项目 | 内容 |
|---|---|
| **客户** | 李女士（铂金会员） |
| **商品** | Barista Pro 智能咖啡机 ¥3,299 |
| **问题** | 外壳开裂 + 水箱漏水（物流挤压导致） |
| **定责** | 物流责任，客户当场拍照，证据链完整 |
| **方案** | ✅ 换新（华东仓有货）+ ✅ ¥80 优惠券补偿 |

---

### 生成的文件

| 文件 | 用途 | 路径 |
|---|---|---|
| **内部处理报告** | 含证据表、政策依据、处理动作、审批要求、SLA | `/after_sales_resolution_report.md` |
| **客户回复草稿** | 礼貌明确、不含风控字段 | `/customer_reply_draft.md` |

### 关键提醒
- **需要等待售后主管审批**（高价值订单规则）后才能执行换新发货和补偿发放
- 客户回复承诺了"今天内"有专人联系，请审批通过后及时跟进

虚拟文件系统 files:
{
  "/after_sales_resolution_report.md": {
    "content": "# 售后工单处理报告\n\n**工单编号**：AS-2026-0710-001  \n**创建时间**：2026-07-10 09:12  \n**优先级**：高  \n**处理人**：系统自动生成\n\n---\n\n## 1. 基本信息\n\n| 字段 | 值 |\n|---|---|\n| 客户 | 李女士（C-10086） |\n| 会员等级 | 铂金（Platinum） |\n| 订单号 | OD-2026-0628-9001 |\n| 商品 | Barista Pro 智能咖啡机（SKU-COFFEE-PRO-01） |\n| 实付金额 | ¥3,299.00 |\n| 支付方式 | 信用卡 |\n| 下单时间 | 2026-06-28 20:18 |\n| 承诺送达日 | 2026-07-03 |\n| 实际签收日 | 2026-07-02 16:40 |\n| 签收方式 | 本人签收 |\n| 理赔渠道 | Ap

## 14. 批量演示：处理多个不同类型工单

这个单元用“大工具”批量跑 3 个工单，方便看业务差异。

这里只跑普通 Python，不额外调用 LLM，所以速度快。


In [15]:
for row in after_sales_data["tickets"]:
    result = json.loads(run_after_sales_review(row["ticket_id"]))
    decision = result["decision"]
    print("=" * 80)
    print("工单:", row["ticket_id"])
    print("客户诉求:", row["customer_message"])
    print("问题类型:", decision["issue_type"])
    print("建议:", decision["recommendation"])
    print("处理动作:", decision["actions"])
    print("阻塞项:", decision["blockers"])
    print("审批:", decision["approvals"])


工单: AS-2026-0710-001
客户诉求: 我买的智能咖啡机到货后发现外壳裂了，水箱也漏水。已经影响使用，希望今天给我一个处理结果。
问题类型: 到货破损
建议: 建议通过，但需要主管审批后执行。
处理动作: ['创建换新单，优先从可用新库存发出。']
阻塞项: []
审批: ['售后主管审批：高价值订单涉及退款/换新/补偿。']
工单: AS-2026-0710-002
客户诉求: 订单显示已经签收，但我没有收到包裹。小区驿站也查不到，请帮忙处理。
问题类型: 未收到货
建议: 暂缓自动处理，先补充核查或人工审批。
处理动作: ['发起物流核查，要求承运商提供签收凭证。']
阻塞项: ['客户风险等级非低，物流显示签收，不能直接退款。']
审批: []
工单: AS-2026-0710-003
客户诉求: 耳机用了两天不喜欢，想退货退款，包装和配件都还在。
问题类型: 七天无理由退货
建议: 建议自动处理。
处理动作: ['创建退货退款单，客户寄回后验货退款。']
阻塞项: []
审批: []


## 15. 程序化验收：检查 Agent 是否真的完成任务

复杂 Agent 不要只看回答像不像。

下面检查：

- 是否调用了预期工具。
- 是否生成了最终回答。
- 是否写了报告文件。
- 回答里是否包含处理动作、审批、SLA 等关键内容。


In [16]:
def inspect_agent_result(agent_result: dict, expected_tools: set[str] | None = None) -> dict:
    """检查 Agent 结果的关键证据。"""
    messages = agent_result.get("messages", [])
    final_answer = messages[-1].content if messages else ""

    called_tools = []
    for message in messages:
        for call in getattr(message, "tool_calls", None) or []:
            called_tools.append(call.get("name"))

    expected_tools = expected_tools or set()
    return {
        "has_final_answer": bool(final_answer.strip()),
        "called_tools": called_tools,
        "missing_expected_tools": sorted(expected_tools - set(called_tools)),
        "has_action_text": "处理" in final_answer or "换新" in final_answer or "退款" in final_answer,
        "has_approval_text": "审批" in final_answer,
        "has_sla_text": "SLA" in final_answer or "响应" in final_answer,
        "files": list((agent_result.get("files") or {}).keys()),
        "final_answer_preview": final_answer[:500],
    }


print("1 个大工具 Agent 验收:")
pprint(inspect_agent_result(simple_result, {"run_after_sales_review"}), width=120)

print("\n多个小工具 Agent 验收:")
pprint(
    inspect_agent_result(
        multi_tool_result,
        {"get_ticket", "get_order", "get_logistics", "get_customer", "get_inventory", "search_policy", "get_sla_rule"},
    ),
    width=120,
)

print("\n报告 Agent 验收:")
pprint(inspect_agent_result(report_result, {"run_after_sales_review"}), width=120)


1 个大工具 Agent 验收:
{'called_tools': ['run_after_sales_review'],
 'files': [],
 'final_answer_preview': '## 工单 AS-2026-0710-001 处理结果\n'
                         '\n'
                         '### 📋 问题类型\n'
                         '**到货破损** — 客户李女士购买的 Barista Pro 智能咖啡机（SKU-COFFEE-PRO-01，金额 3299 '
                         '元）到货后发现外壳裂开、水箱漏水。物流外箱有明显挤压痕迹，客户当场拍照，且在签收后 72 小时内反馈，属于完整的物流破损举证。\n'
                         '\n'
                         '---\n'
                         '\n'
                         '### 💡 处理建议\n'
                         '1. **优先换新** — 华东仓目前有 3 台新库存可用，建议立即创建换新单，从新库存发出。\n'
                         '2. **客户体验补偿** — 因物流破损影响客户体验，建议补偿 80 元优惠券。\n'
                         '3. **退款兜底** — 如换新过程中出现异常，可按照政策原路退款并补偿不超过订单金额的 5%（约 165 元）。\n'
                         '\n'
                         '---\n'
                         '\n'
                         '### ✅ 是否需要审批\n'
                         '**是，需要售后主管审批。** 原因：\n'
                         '- 订单金额 3299 元 > 2000 元，属于高价值订单，涉及

## 16. 真实项目里怎么落地？

推荐架构：

| 模块 | 建议 |
| --- | --- |
| 查订单、查物流、查库存 | 后端 API 工具化，返回 JSON |
| 售后规则 | 尽量由确定性规则引擎执行 |
| Agent | 负责调工具、解释证据、生成报告和回复草稿 |
| 退款、换新、补偿 | 必须二次确认后由后端接口执行 |
| 审计 | 保存工具调用轨迹、最终报告、人工确认人 |

关键原则：

- Agent 可以建议退款，但不要直接退款。
- Agent 可以生成换新方案，但不要直接创建换新单。
- 高价值订单、风险客户、物流签收争议都要人工审批。
- 给客户的回复不要暴露内部风控等级。
